# 🏠 Modelo Predictivo de Precios de Airbnb

**Ciencia de Datos | Gerardo Olivares**

Este notebook desarrolla un modelo de regresión lineal múltiple para predecir el precio de listings de Airbnb en 6 ciudades de Estados Unidos (NYC, LA, SF, Chicago, Boston, DC).

El análisis sigue el flujo completo: limpieza de datos → selección de variables → construcción del modelo → evaluación → predicciones → comunicación de resultados.

---
## 📦 Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('Librerías cargadas correctamente ✓')

---
## Parte 1 — Preparación de Datos para Modelado

### 1.1 Carga del dataset

Cargamos el CSV generado en el avance previo. Se usa `on_bad_lines='skip'` por las comas internas en el campo `amenities`.

In [ ]:
df = pd.read_csv('Base_de_datos_Proyecto.csv', on_bad_lines='skip', low_memory=False)
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)

### 1.2 Conversión de tipos y recuperación del precio

Convertimos columnas numéricas que se importaron como texto y recuperamos el precio real a partir de `log_price`.

In [ ]:
numeric_cols = ['bathrooms', 'bedrooms', 'beds', 'number_of_reviews',
                'review_scores_rating', 'latitude', 'longitude', 'accommodates']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['price'] = np.exp(df['log_price'])

print('Tipos convertidos:')
print(df[numeric_cols + ['log_price', 'price']].dtypes)

### 1.3 Limpieza de datos

#### 1.3.1 Diagnóstico de valores nulos

In [ ]:
nulls = df.isnull().mean().sort_values(ascending=False) * 100
nulls_relevantes = nulls[nulls > 0]

print('Porcentaje de valores nulos por variable:')
print(nulls_relevantes.round(2).to_string())

#### 1.3.2 Estrategia de imputación

Se aplican las siguientes estrategias según el tipo y porcentaje de nulos:

- **`review_scores_rating` (23%)** → imputación con la mediana (distribución sesgada negativamente)
- **`bathrooms`, `bedrooms`, `beds` (<1%)** → imputación con la mediana
- **`number_of_reviews` (<1%)** → imputación con 0 (ausencia de reseñas es información válida)
- **`latitude`, `longitude` (<1%)** → eliminación de filas (no se puede imputar coordenadas)

In [ ]:
# Imputación con mediana
for col in ['review_scores_rating', 'bathrooms', 'bedrooms', 'beds']:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)
    print(f'{col}: imputado con mediana = {mediana}')

# number_of_reviews: imputar con 0
df['number_of_reviews'] = df['number_of_reviews'].fillna(0)
print('number_of_reviews: imputado con 0')

# Eliminar filas sin coordenadas
antes = len(df)
df = df.dropna(subset=['latitude', 'longitude'])
print(f'Filas eliminadas por coordenadas nulas: {antes - len(df)}')

print(f'\nDataset limpio: {df.shape[0]:,} filas')
print(f'Nulos restantes en columnas de interés: {df[numeric_cols].isnull().sum().sum()}')

#### 1.3.3 Identificación y tratamiento de valores atípicos extremos

Se identifican outliers usando el método IQR (rango intercuartílico). Los valores claramente incorrectos (fuera de 3×IQR) se tratan como errores de captura y se eliminan.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, ['price', 'accommodates', 'bathrooms']):
    sns.boxplot(y=df[col], ax=ax, color='steelblue')
    ax.set_title(f'Outliers: {col}')
plt.suptitle('Distribución de variables antes de filtrar outliers', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def remove_outliers_iqr(df, col, factor=3.0):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        print(f'{col}: IQR=0 (variable discreta) — se omite filtro IQR, se eliminan extremos absolutos')
        return df
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    mask = df[col].between(lower, upper)
    removed = (~mask).sum()
    print(f'{col}: rango [{lower:.2f}, {upper:.2f}] — {removed} outliers eliminados')
    return df[mask]

antes = len(df)

# Solo aplicar IQR a variables continuas
for col in ['log_price', 'accommodates']:
    df = remove_outliers_iqr(df, col, factor=3.0)

# Para variables discretas: eliminar valores claramente incorrectos (extremos absolutos)
df = df[df['bathrooms'].between(0, 8)]
df = df[df['bedrooms'].between(0, 8)]
df = df[df['beds'].between(0, 16)]
print(f'bathrooms: limitado a [0, 8]')
print(f'bedrooms:  limitado a [0, 8]')
print(f'beds:      limitado a [0, 16]')

print(f'\nRegistros antes: {antes:,} | después: {len(df):,} | eliminados: {antes - len(df):,}')


---
## 2. Identificación de Variables

### 2.1 Variable dependiente

- **`log_price`** → precio del alojamiento en escala logarítmica. Se usa esta transformación porque el precio original tiene una distribución muy sesgada (skewness > 4). El modelo predecirá `log_price` y las predicciones finales se convierten con `exp()`.

### 2.2 Variables independientes candidatas

Del análisis EDA previo, las variables con mayor correlación con el precio son:

In [ ]:
# Variables categóricas → codificación One-Hot
df_model = df.copy()

# One-hot encoding de variables categóricas relevantes
cat_vars = ['room_type', 'city', 'cancellation_policy']
df_model = pd.get_dummies(df_model, columns=cat_vars, drop_first=True)

# Variables numéricas
num_vars = ['accommodates', 'bathrooms', 'bedrooms', 'beds',
            'number_of_reviews', 'review_scores_rating', 'latitude', 'longitude']

# Variables categóricas codificadas
cat_encoded = [c for c in df_model.columns if any(c.startswith(v+'_') for v in cat_vars)]

all_features = num_vars + cat_encoded
TARGET = 'log_price'

print(f'Variable dependiente: {TARGET}')
print(f'Variables independientes ({len(all_features)}): {all_features}')

---
## 3. Selección de Características

### 3.1 Análisis de correlación con el precio

In [ ]:
corr_num = df[num_vars + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_num.values]
bars = ax.barh(corr_num.index, corr_num.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(0.1, color='gray', linestyle='--', linewidth=0.8, label='Umbral |r|=0.1')
ax.axvline(-0.1, color='gray', linestyle='--', linewidth=0.8)
for bar, val in zip(bars, corr_num.values):
    ax.text(val + 0.01 if val >= 0 else val - 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
ax.set_title('Correlación de variables numéricas con log_price', fontsize=12, fontweight='bold')
ax.set_xlabel('Correlación de Pearson')
ax.legend()
plt.tight_layout()
plt.show()

print(corr_num.round(3))

### 3.2 Selección final de variables

Con base en el análisis de correlación y el EDA previo, se seleccionan las variables con mayor poder explicativo. Se excluye `beds` por alta multicolinealidad con `accommodates` (r=0.81), y `number_of_reviews` por correlación prácticamente nula.

In [ ]:
# Variables seleccionadas para el modelo final
# Se excluyen latitude/longitude porque las variables city ya capturan
# el efecto geográfico a nivel ciudad, evitando multicolinealidad (Cond.No alto).
# Se excluye beds por alta correlación con accommodates (r=0.81).
# Se excluye number_of_reviews por correlación prácticamente nula con log_price.
selected_num = ['accommodates', 'bathrooms', 'bedrooms', 'review_scores_rating']
selected_cat = [c for c in cat_encoded]
FEATURES = selected_num + selected_cat

X = df_model[FEATURES].copy()
y = df_model[TARGET].copy()

# Asegurar que no haya NaN
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]

print(f'Features en el modelo: {len(FEATURES)}')
print(f'Observaciones: {len(X):,}')
print(f'\nFeatures numéricas: {selected_num}')
print(f'Features categóricas codificadas: {selected_cat}')


---
## 4. Análisis de Correlación — Pairplot

Generamos un pairplot con Seaborn para visualizar las relaciones entre las variables numéricas más importantes del modelo y la variable objetivo.

In [ ]:
pairplot_cols = ['log_price', 'accommodates', 'bedrooms', 'bathrooms', 'review_scores_rating']
sample_pp = df[pairplot_cols].dropna().sample(n=2000, random_state=42)

g = sns.pairplot(sample_pp, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10},
                 diag_kws={'color': 'steelblue'}, corner=True)
g.fig.suptitle('Pairplot — Variables numéricas principales vs. log_price', 
               y=1.02, fontsize=13, fontweight='bold')
plt.show()

---
## 5. Hypothesis Testing — Pruebas de hipótesis

Antes de construir el modelo, verificamos estadísticamente si las diferencias de precio entre grupos son significativas.

Se aplica:
- **t-test de Welch** (dos grupos independientes, varianzas desiguales): compara el log-precio entre *Entire home/apt* y *Private room*
- **ANOVA de una vía**: compara el log-precio entre las 6 ciudades

**H₀:** No existe diferencia estadísticamente significativa entre los grupos  
**H₁:** Existe diferencia estadísticamente significativa (p < 0.05)

In [ ]:
from scipy.stats import ttest_ind, f_oneway

# ── T-TEST: Entire home/apt vs Private room ──────────────────────────────────
entire  = df[df['room_type'] == 'Entire home/apt']['log_price']
private = df[df['room_type'] == 'Private room']['log_price']

t_stat, p_val = ttest_ind(entire, private, equal_var=False)  # Welch t-test

print('='*55)
print('T-TEST: Entire home/apt vs Private room')
print('='*55)
print(f't-statistic: {t_stat:.4f}')
print(f'p-value:     {p_val:.2e}')
if p_val < 0.05:
    print('→ Existe una diferencia estadísticamente significativa')
    print(f'  (p < 0.05). El log-precio de Entire home/apt')
    print(f'  ({entire.mean():.3f}) es mayor que Private room ({private.mean():.3f}).')
else:
    print('→ No existe diferencia significativa (p >= 0.05)')

# ── ANOVA: diferencias entre ciudades ────────────────────────────────────────
print('\n' + '='*55)
print('ANOVA: Diferencias de precio entre ciudades')
print('='*55)
grupos_ciudad = [df[df['city'] == c]['log_price'] for c in df['city'].unique()]
f_stat, p_anova = f_oneway(*grupos_ciudad)

print(f'F-statistic: {f_stat:.4f}')
print(f'p-value:     {p_anova:.2e}')
if p_anova < 0.05:
    print('→ Existe diferencia estadísticamente significativa')
    print('  entre al menos un par de ciudades (p < 0.05).')
    print('  Esto valida incluir \'city\' como feature en el modelo.')
else:
    print('→ No existe diferencia significativa entre ciudades (p >= 0.05)')

# ── Visualización ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# t-test visual
data_tt = df[df['room_type'].isin(['Entire home/apt', 'Private room'])]
sns.boxplot(data=data_tt, x='room_type', y='log_price',
            palette='Set2', ax=axes[0], width=0.4)
axes[0].set_title(f'T-Test: Entire vs Private\np-value = {p_val:.2e}  ✓ Significativo',
                  fontweight='bold')
axes[0].set_xlabel('Tipo de habitación')
axes[0].set_ylabel('log(precio)')

# ANOVA visual
city_order = df.groupby('city')['log_price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='city', y='log_price', order=city_order,
            palette='Set3', ax=axes[1], width=0.4)
axes[1].set_title(f'ANOVA: Log-precio por ciudad\np-value = {p_anova:.2e}  ✓ Significativo',
                  fontweight='bold')
axes[1].set_xlabel('Ciudad')
axes[1].set_ylabel('')

plt.suptitle('Hypothesis Testing — Diferencias de precio entre grupos',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. División en Conjuntos de Entrenamiento y Prueba

Se divide el dataset en 80% para entrenamiento y 20% para prueba, usando semilla aleatoria fija para reproducibilidad.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Conjunto de entrenamiento: {X_train.shape[0]:,} filas ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Conjunto de prueba:        {X_test.shape[0]:,} filas ({X_test.shape[0]/len(X)*100:.1f}%)')

# Verificar distribución de y en ambos conjuntos
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
axes[0].hist(y_train, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Distribución log_price — Entrenamiento')
axes[1].hist(y_test, bins=40, color='seagreen', alpha=0.8, edgecolor='white')
axes[1].set_title('Distribución log_price — Prueba')
for ax in axes:
    ax.set_xlabel('log(precio)')
plt.tight_layout()
plt.show()

print(f'\nMedia log_price — Train: {y_train.mean():.4f} | Test: {y_test.mean():.4f}')

---
## 7. Construcción y Entrenamiento del Modelo

### 7.1 Análisis de multicolinealidad (VIF)

Antes de entrenar, verificamos el Factor de Inflación de Varianza (VIF) para detectar multicolinealidad entre variables. Un VIF > 10 indica multicolinealidad problemática.

In [ ]:
# Calcular VIF solo para variables numéricas
X_vif = sm.add_constant(X_train[selected_num].astype(float))
vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data[vif_data['Variable'] != 'const'].sort_values('VIF', ascending=False)

fig, ax = plt.subplots(figsize=(7, 3))
colors = ['tomato' if v > 10 else 'steelblue' for v in vif_data['VIF']]
ax.barh(vif_data['Variable'], vif_data['VIF'], color=colors)
ax.axvline(10, color='red', linestyle='--', label='Umbral VIF=10')
ax.axvline(5, color='orange', linestyle='--', alpha=0.6, label='Umbral VIF=5')
ax.set_title('Factor de Inflación de Varianza (VIF)', fontsize=12, fontweight='bold')
ax.set_xlabel('VIF')
ax.legend()
plt.tight_layout()
plt.show()

print(vif_data.to_string(index=False))
print('\nTodas las variables tienen VIF < 5: no hay multicolinealidad problemática.')


### 7.2 Entrenamiento del modelo de regresión lineal múltiple

Se entrena un modelo de Regresión Lineal Múltiple (OLS) con las variables seleccionadas. Se usa `statsmodels` para obtener un resumen estadístico detallado con p-values y coeficientes, y `scikit-learn` para predicciones.

In [ ]:
# Modelo con scikit-learn para métricas y predicciones
# Sin necesidad de escalar: todas las variables numéricas están en escala comparable
modelo = LinearRegression()
modelo.fit(X_train, y_train)

print('Modelo entrenado correctamente ✓')
print(f'Intercepto: {modelo.intercept_:.4f}')
print(f'Número de coeficientes: {len(modelo.coef_)}')


In [ ]:
# Modelo con statsmodels para resumen estadístico completo
X_train_sm = sm.add_constant(X_train.astype(float))
modelo_sm = sm.OLS(y_train.astype(float), X_train_sm).fit()
print(modelo_sm.summary())


### 7.3 Coeficientes del modelo

Visualizamos los coeficientes más importantes del modelo. En escala log, un coeficiente de 0.1 implica que la variable aumenta el precio en ~10.5%.

In [ ]:
coef_df = pd.DataFrame({'Feature': FEATURES, 'Coeficiente': modelo.coef_})
coef_df['|Coeficiente|'] = coef_df['Coeficiente'].abs()
coef_df = coef_df.sort_values('|Coeficiente|', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['steelblue' if v > 0 else 'tomato' for v in coef_df['Coeficiente']]
ax.barh(coef_df['Feature'], coef_df['Coeficiente'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes del modelo (Top 20 por magnitud)', fontsize=12, fontweight='bold')
ax.set_xlabel('Valor del coeficiente (escala log-precio)')
plt.tight_layout()
plt.show()


---
## 8. Evaluación del Modelo

### 8.1 Métricas sobre el conjunto de prueba

In [ ]:
y_pred_train = modelo.predict(X_train)
y_pred_test  = modelo.predict(X_test)

r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test, y_pred_test)
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

# Convertir errores a escala de precio real
precio_real  = np.exp(y_test)
precio_pred  = np.exp(y_pred_test)
mae_dolares  = mean_absolute_error(precio_real, precio_pred)
rmse_dolares = np.sqrt(mean_squared_error(precio_real, precio_pred))

print('='*50)
print('       MÉTRICAS DE EVALUACIÓN DEL MODELO')
print('='*50)
print(f'R² (entrenamiento):          {r2_train:.4f}')
print(f'R² (prueba):                 {r2_test:.4f}')
print(f'MSE (prueba, log_price):     {mse_test:.4f}')
print(f'RMSE (prueba, log_price):    {rmse_test:.4f}')
print(f'MAE (prueba, log_price):     {mae_test:.4f}')
print('-'*50)
print(f'MAE  en dólares reales:      ${mae_dolares:.2f}')
print(f'RMSE en dólares reales:      ${rmse_dolares:.2f}')
print('='*50)
print(f'\nInterpretación R²: el modelo explica el {r2_test*100:.1f}% de la')
print(f'varianza del log-precio en datos no vistos.')


### 8.2 Análisis de residuales

Los residuales deben ser aleatorios y aproximadamente normales para que el modelo sea válido. Verificamos tres supuestos: normalidad, homocedasticidad y ausencia de patrón sistemático.

In [ ]:
residuales = y_test - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Residuales vs predicciones
axes[0].scatter(y_pred_test, residuales, alpha=0.3, s=8, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Valores predichos (log_price)')
axes[0].set_ylabel('Residuales')
axes[0].set_title('Residuales vs. Predichos')

# 2. Distribución de residuales
axes[1].hist(residuales, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_title(f'Distribución de Residuales\n(media={residuales.mean():.3f}, std={residuales.std():.3f})')

# 3. QQ-plot
from scipy import stats
stats.probplot(residuales, dist='norm', plot=axes[2])
axes[2].set_title('QQ-Plot de Residuales')
axes[2].get_lines()[1].set_color('red')

plt.suptitle('Diagnóstico de Residuales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Predicciones

### 9.1 Predicciones vs. valores reales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Escala log_price ---
lim_min = min(y_test.min(), y_pred_test.min()) - 0.2
lim_max = max(y_test.max(), y_pred_test.max()) + 0.2
axes[0].scatter(y_test, y_pred_test, alpha=0.25, s=8, color='steelblue')
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('log(precio) real')
axes[0].set_ylabel('log(precio) predicho')
axes[0].set_title(f'Predicciones vs. Reales (log_price)\nR² = {r2_test:.4f}')
axes[0].legend()
axes[0].set_xlim(lim_min, lim_max)
axes[0].set_ylim(lim_min, lim_max)

# --- Escala precio real (dólares) ---
precio_real_plot = np.exp(y_test)
precio_pred_plot = np.exp(y_pred_test)
mask_plot = precio_real_plot < 800
axes[1].scatter(precio_real_plot[mask_plot], precio_pred_plot[mask_plot],
                alpha=0.25, s=8, color='seagreen')
lim2 = 800
axes[1].plot([0, lim2], [0, lim2], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[1].set_xlabel('Precio real (USD)')
axes[1].set_ylabel('Precio predicho (USD)')
axes[1].set_title(f'Predicciones vs. Reales (USD, <$800)\nMAE = ${mae_dolares:.2f}')
axes[1].legend()

plt.suptitle('Comparación de Predicciones vs. Valores Reales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.2 Ejemplos de predicciones individuales

In [ ]:
# Muestra de 10 predicciones aleatorias
sample_idx = np.random.RandomState(42).choice(len(y_test), 10, replace=False)
y_test_arr  = y_test.values[sample_idx]
y_pred_arr  = y_pred_test[sample_idx]

comparacion = pd.DataFrame({
    'log_price real':     y_test_arr.round(3),
    'log_price predicho': y_pred_arr.round(3),
    'Precio real ($)':    np.exp(y_test_arr).round(2),
    'Precio predicho ($)':np.exp(y_pred_arr).round(2),
    'Error ($)':          (np.exp(y_pred_arr) - np.exp(y_test_arr)).round(2),
    'Error %':            ((np.exp(y_pred_arr) - np.exp(y_test_arr)) / np.exp(y_test_arr) * 100).round(1)
})

print('Muestra de 10 predicciones individuales:')
print(comparacion.to_string())

---
## 10. Cálculo del Error Cuadrático Medio (MSE)

El MSE mide el promedio de los errores al cuadrado. Se interpreta en la escala de la variable objetivo (log_price). Un MSE bajo indica mejor ajuste.

In [ ]:
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)

print('='*55)
print('   ERROR CUADRÁTICO MEDIO (MSE) — RESUMEN FINAL')
print('='*55)
print(f'MSE  (log_price):     {mse:.6f}')
print(f'RMSE (log_price):     {rmse:.6f}')
print(f'MAE  (log_price):     {mae_test:.6f}')
print('-'*55)
print(f'MSE  (dólares):       {mean_squared_error(precio_real, precio_pred):.2f}')
print(f'RMSE (dólares):       ${rmse_dolares:.2f}')
print(f'MAE  (dólares):       ${mae_dolares:.2f}')
print('='*55)
print(f'\nEl modelo tiene un error medio absoluto de ${mae_dolares:.2f} USD.')
print(f'Dado que el precio mediano es ~$110, esto representa un')
print(f'error relativo de ~{mae_dolares/110*100:.1f}% sobre la mediana del mercado.')

---
## Parte 2 — Comunicación de Resultados

### 11.1 Importancia de características

Visualizamos el impacto de cada variable en el precio predicho usando los coeficientes del modelo en escala log.

In [ ]:
coef_full = pd.DataFrame({'Feature': FEATURES, 'Coeficiente': modelo.coef_})
coef_full['Impacto_%'] = (np.exp(coef_full['Coeficiente']) - 1) * 100
coef_full = coef_full.reindex(coef_full['Coeficiente'].abs().sort_values(ascending=True).index)
top20 = coef_full.tail(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Coeficientes
colors = ['steelblue' if v > 0 else 'tomato' for v in top20['Coeficiente']]
axes[0].barh(top20['Feature'], top20['Coeficiente'], color=colors)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Coeficientes del Modelo\n(Top 20 por magnitud)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Coeficiente (log_price)')

# Impacto en % de precio
colors2 = ['steelblue' if v > 0 else 'tomato' for v in top20['Impacto_%']]
axes[1].barh(top20['Feature'], top20['Impacto_%'], color=colors2)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Impacto Estimado sobre el Precio\n(% de cambio)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Cambio % en precio (ceteris paribus)')

plt.suptitle('Importancia de Variables en el Modelo de Regresión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 11.2 Distribución del error por ciudad y tipo de habitación

Analizamos si el modelo comete errores sistemáticos en algún segmento del mercado.

In [ ]:
# Reconstruir contexto de los datos de prueba
df_test_ctx = df_model.loc[X_test.index].copy()
df_test_ctx['y_real'] = y_test.values
df_test_ctx['y_pred'] = y_pred_test
df_test_ctx['error']  = df_test_ctx['y_real'] - df_test_ctx['y_pred']

# Recuperar ciudad y room_type del df original
df_test_ctx['city_orig']      = df.loc[X_test.index, 'city'].values
df_test_ctx['room_type_orig'] = df.loc[X_test.index, 'room_type'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

city_order = df_test_ctx.groupby('city_orig')['error'].median().abs().sort_values().index
sns.boxplot(data=df_test_ctx, x='city_orig', y='error', order=city_order,
            palette='Set2', ax=axes[0], width=0.5)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Error del modelo por ciudad', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Ciudad')
axes[0].set_ylabel('Residual (log_price real - predicho)')

sns.boxplot(data=df_test_ctx, x='room_type_orig', y='error',
            palette='Set3', ax=axes[1], width=0.5)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Error del modelo por tipo de habitación', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Tipo de habitación')
axes[1].set_ylabel('')

plt.suptitle('Distribución del Error por Segmento de Mercado', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Error medio (residual) por ciudad:')
print(df_test_ctx.groupby('city_orig')['error'].agg(['mean','std']).round(4))

### 11.3 Predicciones por rango de precio real

Visualizamos cómo se comporta el modelo en diferentes rangos de precio para identificar dónde es más y menos preciso.

In [ ]:
df_test_ctx['precio_real']  = np.exp(df_test_ctx['y_real'])
df_test_ctx['precio_pred']  = np.exp(df_test_ctx['y_pred'])
df_test_ctx['error_dolares']= (df_test_ctx['precio_pred'] - df_test_ctx['precio_real']).abs()

bins = [0, 75, 125, 200, 350, 99999]
labels = ['<$75', '$75-125', '$125-200', '$200-350', '>$350']
df_test_ctx['rango_precio'] = pd.cut(df_test_ctx['precio_real'], bins=bins, labels=labels)

resumen_rango = df_test_ctx.groupby('rango_precio', observed=True).agg(
    n=('error_dolares', 'count'),
    MAE_dolares=('error_dolares', 'mean'),
    R2_aprox=('y_real', lambda x: r2_score(x, df_test_ctx.loc[x.index, 'y_pred']))
).round(3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

resumen_rango['MAE_dolares'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('MAE (USD) por Rango de Precio', fontweight='bold')
axes[0].set_xlabel('Rango de precio real')
axes[0].set_ylabel('MAE ($)')
axes[0].tick_params(axis='x', rotation=30)

resumen_rango['R2_aprox'].plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('R² por Rango de Precio', fontweight='bold')
axes[1].set_xlabel('Rango de precio real')
axes[1].set_ylabel('R²')
axes[1].axhline(r2_test, color='red', linestyle='--', label=f'R² global={r2_test:.3f}')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Precisión del Modelo por Rango de Precio', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(resumen_rango.to_string())

---
## 12. Narrativa y Conclusiones del Modelo

### 12.1 Resumen ejecutivo

In [ ]:
print('='*60)
print('   RESUMEN FINAL DEL MODELO')
print('='*60)
print(f'Modelo:              Regresión Lineal Múltiple (OLS)')
print(f'Variable objetivo:   log_price (precio logarítmico)')
print(f'Features:            {len(FEATURES)} variables')
print(f'Observaciones train: {X_train.shape[0]:,}')
print(f'Observaciones test:  {X_test.shape[0]:,}')
print('-'*60)
print(f'R² (test):           {r2_test:.4f}  →  explica {r2_test*100:.1f}% de la varianza')
print(f'RMSE (log_price):    {rmse:.4f}')
print(f'MSE  (log_price):    {mse:.4f}')
print(f'MAE  ($USD):         ${mae_dolares:.2f} promedio de error en precio real')
print('-'*60)
print('VARIABLES MÁS INFLUYENTES:')
top5 = coef_full.tail(5)[['Feature','Coeficiente','Impacto_%']].iloc[::-1]
for _, row in top5.iterrows():
    signo = '+' if row['Coeficiente'] > 0 else ''
    print(f'  {row["Feature"]:<35} coef={signo}{row["Coeficiente"]:.3f}  ({signo}{row["Impacto_%"]:.1f}% en precio)')
print('='*60)

### Conclusiones

**1. Calidad del modelo**  
El modelo de regresión lineal múltiple alcanza un **R² = 0.536** en el conjunto de prueba, explicando el 53.6% de la varianza del log-precio con 15 variables (4 numéricas + 11 categóricas codificadas). Para un modelo lineal aplicado a un mercado tan heterogéneo — propiedades desde $30 hasta $2,000 por noche — este resultado es razonable. El error medio absoluto es **$58.72 USD**, que sobre un precio mediano de $110 representa un error relativo del ~53%. Esto indica que el modelo captura la tendencia general del precio pero no los factores más finos de cada propiedad.

**2. Variables más influyentes — Implicaciones de negocio**  
- **The strongest numerical predictor of price is guest capacity** (`accommodates`, r=0.57, coef=+0.075): cada huésped adicional de capacidad aumenta el precio estimado ~7.8%. Los hosts deberían maximizar la capacidad de su propiedad si buscan incrementar ingresos.
- **Room type es el driver categórico más crítico**: una habitación privada reduce el precio estimado **45.8%** respecto a un alojamiento completo (coef=-0.613, exp(-0.613)-1=-45.8%). Una habitación compartida lo reduce aún más (-65.2%). Los hosts deberían ofrecer el espacio completo cuando sea posible.
- **La ciudad determina el nivel base de precios**: tomando Boston como referencia, SF cobra **33.6% más** (coef=+0.290), DC un 3.3% más, mientras que NYC cobra 2.2% menos, LA un 14.5% menos y Chicago un 26.7% menos. El ANOVA (F=572.07, p≈0.00) confirma que estas diferencias son altamente significativas.
- **La calificación tiene un efecto positivo pequeño pero estadísticamente significativo** (coef=+0.0076, p<0.05): cada punto adicional en `review_scores_rating` aumenta el precio ~0.76%. Mantener una buena reputación tiene valor económico real aunque modesto.
- **Políticas de cancelación estrictas asociadas a precios más altos**: `super_strict_60` tiene el coef más alto entre políticas (+0.834, +130% de precio), reflejando que propiedades de lujo adoptan estas políticas — no que la política cause el precio.

**3. Hypothesis Testing — Validación estadística**  
- El **t-test de Welch** (t=187.16, p≈0.00) confirma que la diferencia de precio entre *Entire home/apt* (log_price medio = 5.15) y *Private room* (log_price medio = 4.33) es estadísticamente significativa. En términos de precio real, un alojamiento completo cuesta en promedio **~128% más** que una habitación privada comparable.
- El **ANOVA de una vía** (F=572.07, p≈0.00) confirma diferencias altamente significativas entre las 6 ciudades, validando `city` como variable de control esencial en el modelo.

**4. Comportamiento del modelo por segmento**  
- El modelo no presenta sesgo geográfico: el residual medio por ciudad es menor a 0.03 en todas las ciudades, con Boston mostrando el mayor sesgo (0.021).
- El MAE en dólares escala con el rango de precio: propiedades <$75 tienen MAE=$24, mientras que propiedades >$350 tienen MAE=$357. Esto es esperado — factores de lujo no capturados en las variables generan errores mayores en el segmento premium.
- El R² negativo por rango de precio indica que dentro de cada segmento estrecho el modelo predice peor que usar la media del segmento. Esto es una limitación estructural del modelo lineal en mercados segmentados y confirma la necesidad de modelos más complejos.

**5. Limitaciones y mejoras futuras**  
- El R²=0.536 indica que el 46.4% de la varianza del precio no es explicada por el modelo lineal. Modelos como **Random Forest o Gradient Boosting** capturarían interacciones no lineales entre variables (p.ej. el efecto de `accommodates` varía según `room_type`) y mejorarían significativamente el desempeño.
- El campo `amenities` contiene información valiosa que podría procesarse con NLP (indicadores binarios: 'tiene piscina', 'tiene estacionamiento', 'tiene gimnasio').
- La variable `neighbourhood` (9.5% de nulos) aportaría información geográfica intra-ciudad más granular, capturando diferencias de precio entre barrios dentro de la misma ciudad.
- El Condition Number del modelo es 6,760, lo que sugiere cierta multicolinealidad residual. Técnicas como Ridge Regression podrían mejorar la estabilidad numérica.


---
*Gerardo Olivares | Ciencia de Datos*